## From PDBx/mmCIF to Geometric Graph Tensors

**Background**  
In the previous notebooks of Module 1, we extracted $\text{C}_\alpha$ coordinates (Notebook 1), converted spatial coordinates into PyTorch Geometric COO-compatible adjacency tensors (Notebook 2), and constructed fully featured `torch_geometric.data.Data` objects for Graph Neural Network (GNN) modeling (Notebook 3). In this notebook, we execute an end-to-end validation pipeline on the multi-chain **aPKC–Par-6–Lgl** complex (*8r3y.cif*).

**Goals**
1. Process *8r3y.cif* end-to-end to bridge raw structural bioinformatics data with geometric deep learning.
2. Demonstrate how raw 3D Cartesian coordinates, spatial edge topology, and surface physics ($\text{rSASA}$) translate into PyTorch Geometric tensors.

In [12]:
import freesasa
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv
import warnings

from Bio.PDB.PDBExceptions import PDBConstructionWarning
from polarity_engine.parsers import StructureParser as parsers
from polarity_engine.builder import ProteinGraphBuilder as builder

freesasa.setVerbosity(freesasa.silent)
warnings.filterwarnings("ignore", category=PDBConstructionWarning)

In [2]:
data_dir = Path("../../data")
data_dir.mkdir(parents=True, exist_ok=True)

input_dir = data_dir / "input"
input_dir.mkdir(parents=True, exist_ok=True)

### 1. Parse Structure

#### PyG Data Tensor Contract

| Tensor | Shape | Description / Contents |
| :--- | :--- | :--- |
| **`data.x`** | `[N, 25]` | **21** 1-hot AAs + **1** normalized seq pos + **1** rSASA + **1** B-factor + **1** occupancy |
| **`data.pos`** | `[N, 3]` | 3D backbone $\text{C}_\alpha$ Cartesian coordinates ($\text{\AA}$) |
| **`data.edge_index`** | `[2, E]` | Directed spatial graph connectivity matrix |
| **`data.edge_attr`** | `[E, 20]` | **3** normalized unit vectors + **16** Gaussian RBF channels + **1** inter-chain flag |

In [ ]:
TEST_CIF_Paht = input_dir / "8r3y.cif"

parsed_data = parsers.parse(TEST_CIF_Paht)

In [4]:
assert len(parsed_data["aa_list"]) == parsed_data["coords"].shape[0] == len(parsed_data["nodes"]), \
    "Array dimension mismatch across residues, coordinates, and nodes!"

assert not np.isnan(parsed_data["coords"]).any(
), "NaN values found in coordinates!"
assert parsed_data[
    "coords"].dtype == np.float32, f"Expected float32, got {parsed_data['coords'].dtype}"

assert len(parsed_data["sasa_map"]) > 0, "SASA map is empty!"

sample_chain, sample_res_num, sample_res_name = parsed_data["nodes"][0]
assert (sample_chain,
        sample_res_num) in parsed_data["sasa_map"], f"Node ({sample_chain}, {sample_res_num}) missing from SASA map!"

print(f"✅ Data integrity verified for {len(parsed_data['aa_list'])} residues!")
print(
    f"Chains present: {sorted(list(set(node[0] for node in parsed_data['nodes'])))}")

✅ Data integrity verified for 1290 residues!
Chains present: ['I', 'L', 'P']


In [5]:
# Quick sanity check on the parsed structure dictionary
print("Coords shape:     ", parsed_data["coords"].shape)
print("Residues count:   ", len(parsed_data["aa_list"]))
print("Sample residues:  ", parsed_data["aa_list"][:5])
print("B-factors shape:  ", parsed_data["b_factors"].shape)

Coords shape:      (1290, 3)
Residues count:    1290
Sample residues:   ['SER', 'LEU', 'GLY', 'LEU', 'GLN']
B-factors shape:   (1290,)


In [ ]:
# Initialize graph builder
builder = builder()

# Construct PyG Data object from parsed structure dictionary
data = builder.build_graph(
    aa_list=parsed_data["aa_list"],
    coords_np=parsed_data["coords"],
    b_factors_np=parsed_data["b_factors"],
    occupancies_np=parsed_data["occupancies"],
    nodes=parsed_data["nodes"],
    sasa_map=parsed_data["sasa_map"],
    name="8r3y")

# Sanity check PyG Data object outputs
print("PyG Data object:", data)
print("\n--- Node & Edge Tensors ---")
print("x (Node Features):       ", data.x.shape)
print("edge_index (Spatial):    ", data.edge_index.shape)
print("edge_attr (RBF Distance):", data.edge_attr.shape)
if hasattr(data, "sasa"):
    print("sasa (Relative SASA):    ", data.sasa.shape)

PyG Data object: Data(x=[1290, 25], edge_index=[2, 12812], edge_attr=[12812, 20], pos=[1290, 3], name='8r3y')

--- Node & Edge Tensors ---
x (Node Features):        torch.Size([1290, 25])
edge_index (Spatial):     torch.Size([2, 12812])
edge_attr (RBF Distance): torch.Size([12812, 20])


### Feature Distribution Inspection

In [ ]:
print("--- Node Feature Distribution Summary (8r3y) ---")
print(f"Total Nodes (N): {data.x.shape[0]}")

# Feature Column Indexing
seq_pos = data.x[:, 21]
rsasa = data.x[:, 22]
b_factors = data.x[:, 23]
occupancies = data.x[:, 24]

print(
    f"\nSequence Pos (col 21) -> Min: {seq_pos.min():.3f}, Max: {seq_pos.max():.3f}, Mean: {seq_pos.mean():.3f}")
print(
    f"rSASA        (col 22) -> Min: {rsasa.min():.3f}, Max: {rsasa.max():.3f}, Mean: {rsasa.mean():.3f}")
print(
    f"B-factor     (col 23) -> Min: {b_factors.min():.3f}, Max: {b_factors.max():.3f}, Mean: {b_factors.mean():.3f}")
print(
    f"Occupancy    (col 24) -> Min: {occupancies.min():.3f}, Max: {occupancies.max():.3f}, Mean: {occupancies.mean():.3f}")

# Sanity bounds check
assert (seq_pos >= 0.0).all() and (seq_pos <= 1.0).all(
), "Sequence position out of [0, 1] bounds!"
assert (rsasa >= 0.0).all() and (
    rsasa <= 1.0).all(), "rSASA out of [0, 1] bounds!"
print("\nAll feature bounds verified successfully.")

--- Node Feature Distribution Summary (8r3y) ---
Total Nodes (N): 1290

Sequence Pos (col 21) -> Min: 0.000, Max: 1.000, Mean: 0.500
rSASA        (col 22) -> Min: 0.000, Max: 1.000, Mean: 0.222
B-factor     (col 23) -> Min: 8.340, Max: 99.110, Mean: 47.828
Occupancy    (col 24) -> Min: 1.000, Max: 1.000, Mean: 1.000

All feature bounds verified successfully.


### Model Layer Integration Test

In [13]:
class ProteinFeatureEncoder(nn.Module):

  def __init__(self, node_in_dim=25, edge_in_dim=20, hidden_dim=64):
    super().__init__()

    # Initial linear projection for raw 25-dim node features
    self.node_proj = nn.Linear(node_in_dim, hidden_dim)

    # MLP to process raw 20-dim edge attributes into hidden_dim
    self.edge_mlp = nn.Sequential(
        nn.Linear(edge_in_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim),
    )

    # Message-passing neural network for GINEConv
    self.node_mlp = nn.Sequential(
        nn.Linear(hidden_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim),
    )

    # GINEConv expects node features and edge attributes to have matching dimensions (hidden_dim)
    self.conv = GINEConv(nn=self.node_mlp, train_eps=True)

  def forward(self, x, edge_index, edge_attr):
    # 1. Project node features: [1290, 25] -> [1290, 64]
    x_proj = self.node_proj(x)

    # 2. Project edge features: [12812, 20] -> [12812, 64]
    edge_embedding = self.edge_mlp(edge_attr)

    # 3. Message passing: aggregation over spatial graph topology
    out = self.conv(x_proj, edge_index, edge_attr=edge_embedding)
    return out


# Instantiate model and execute forward pass
encoder = ProteinFeatureEncoder(node_in_dim=25, edge_in_dim=20, hidden_dim=64)
output_embeddings = encoder(data.x, data.edge_index, data.edge_attr)

print("--- Model Forward Pass Validation ---")
print("Input Node Features:   ", data.x.shape)
print("Output Node Embeddings:", output_embeddings.shape)
assert output_embeddings.shape == (1290, 64), (
    "Forward pass output shape mismatch!"
)
print("Forward pass executed successfully with no dimension errors.")

--- Model Forward Pass Validation ---
Input Node Features:    torch.Size([1290, 25])
Output Node Embeddings: torch.Size([1290, 64])
Forward pass executed successfully with no dimension errors.


In [14]:
# Create a dummy batch with 2 copies of 8r3y data
batch_list = [data, data]
loader = DataLoader(batch_list, batch_size=2, shuffle=False)

for batch in loader:
    print("\n--- PyG DataLoader Batch Summary ---")
    print("Batch object:              ", batch)
    print("Total Nodes (N1 + N2):     ", batch.x.shape[0])       # Expect 2580
    print("Total Edges (E1 + E2):     ", batch.edge_index.shape[1]) # Expect 25624
    print("Batch Assignment Index:    ", batch.batch.shape[0])   # Expect 2580
    
    # Verify graph batch allocation
    assert batch.x.shape[0] == 2580
    assert batch.edge_index.shape[1] == 25624
    assert torch.equal(torch.unique(batch.batch), torch.tensor([0, 1]))
    print("Mini-batching verified successfully.")


--- PyG DataLoader Batch Summary ---
Batch object:               DataBatch(x=[2580, 25], edge_index=[2, 25624], edge_attr=[25624, 20], pos=[2580, 3], name=[2], batch=[2580], ptr=[3])
Total Nodes (N1 + N2):      2580
Total Edges (E1 + E2):      25624
Batch Assignment Index:     2580
Mini-batching verified successfully.


### Protein Graph Pipeline Status

[Structure File (.cif / .pdb)]  
│  
▼  
[Parsing & Feature Extraction]  
    ├── Backbone Cα Coordinates (pos) [N, 3]  
    ├── 21 One-hot Amino Acid Types  
    ├── Normalized Sequence Position [0, 1]  
    ├── Relative SASA (rSASA) [0, 1]  
    ├── B-factors & Occupancies  
    └── Spatial Edge Attributes (Unit Vectors + RBF + Inter-chain) [E, 20]  
│  
▼  
[PyG Data Object Construction] ──► Verified (8r3y: 1290 nodes, 12812 edges)  
│  
▼  
[GNN Message-Passing Layer]     ──► Verified (GINEConv output: [1290, 64])  
│  
▼ 
[Mini-Batch DataLoader]          ──► Verified (DataBatch: 2580 nodes, 25624 edges)  

#### Tensor Contract Validation Matrix

| Component | Shape | Status | Description |
| :--- | :--- | :---: | :--- |
| **`data.x`** | `[N, 25]` | ✅ Verified | 21 AA + 1 Pos + 1 rSASA + 1 B-factor + 1 Occupancy |
| **`data.pos`** | `[N, 3]` | ✅ Verified | 3D Backbone $\text{C}_\alpha$ Cartesian coordinates ($\text{\AA}$) |
| **`data.edge_index`** | `[2, E]` | ✅ Verified | Directed spatial graph topology |
| **`data.edge_attr`** | `[E, 20]` | ✅ Verified | 3 Unit vectors + 16 Gaussian RBFs + 1 Inter-chain flag |
| **Encoder Forward Pass** | `[N, 64]` | ✅ Verified | $25 \to 64$ node & $20 \to 64$ edge projection via `GINEConv` |
| **Disjoint Mini-Batching** | `[B*N, 25]` | ✅ Verified | PyG `DataLoader` batch allocation without index collisions |